<center>
<img src="../../img/ods_stickers.jpg">
## [mlcourse.ai](https://mlcourse.ai) - دورة التعلم الآلي المفتوحة
    
المؤلفون: [إيليا باريشنيكوف](https://www.linkedin.com/in/baryshnikov-ilya/)، [مكسيم أوفاروف](https://www.linkedin.com/in/maxis42/)، و[يوري كاشنيتسكي](https://www.linkedin.com/in/festline/). تمت الترجمة والتحرير بواسطة [إنجا كايدانوفا](https://www.linkedin.com/in/inga-kaidanova-a92398b1/)، و[إيجور بولوسماك](https://www.linkedin.com/in/egor-polusmak/)، و[أناستازيا مانوخينا](https://www.linkedin.com/in/anastasiamanokhina/)، و[يوانيوان باو](https://www.linkedin.com/in/yuanyuanpao/). يتم توزيع كل المحتوى بموجب ترخيص [Creative Commons CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/).



# <center>المهمة رقم 2 (تجريبي)
## <center>تحليل بيانات أمراض القلب والأوعية الدموية 
    
    
** نفس المهمة مثل [Kaggle Kernel](https://www.kaggle.com/kashnitsky/a2-demo-analyzing-cardiovascular-data) + [الحل](https://www.kaggle.com/kashnitsky/a2-demo-analyzing-cardiovascular-data-solution).**



في هذا التدريب، سوف تجيب على الأسئلة المتعلقة بمجموعة البيانات المتعلقة بأمراض القلب والأوعية الدموية. لا تحتاج إلى تنزيل البيانات: فهي موجودة بالفعل في المستودع. هناك بعض المهام التي ستتطلب منك كتابة التعليمات البرمجية. أكملها ثم أجب عن الأسئلة في [النموذج](https://docs.google.com/forms/d/13cE_tSIb6hsScQvvWUJeu1MEHE5L6vnxQUbDYpXsf24).
#### مشكلة
التنبؤ بوجود أو عدم وجود أمراض القلب والأوعية الدموية (CVD) باستخدام نتائج فحص المريض.
#### وصف البيانات
هناك 3 أنواع من ميزات الإدخال:
- *الهدف*: معلومات واقعية؛
- *الفحص*: نتائج الفحص الطبي؛
- *ذاتي*: المعلومات المقدمة من قبل المريض.| ميزة | نوع متغير | متغير | نوع القيمة |
|---------|---------------------|------------|------------|
| العمر | ميزة الهدف | العمر | كثافة العمليات (أيام) |
| الارتفاع | ميزة الهدف | الارتفاع | كثافة العمليات (سم) |
| الوزن | ميزة الهدف | الوزن | تعويم (كجم) |
| الجنس | ميزة الهدف | الجنس | الكود الفئوي |
| ضغط الدم الانقباضي | ميزة الفحص | ap_hi | كثافة العمليات |
| ضغط الدم الانبساطي | ميزة الفحص | ap_lo | كثافة العمليات |
| الكولسترول | ميزة الفحص | الكولسترول | 1: عادي، 2: أعلى من الطبيعي، 3: أعلى بكثير من الطبيعي |
| الجلوكوز | ميزة الفحص | جلوك | 1: عادي، 2: أعلى من الطبيعي، 3: أعلى بكثير من الطبيعي |
| التدخين | ميزة ذاتية | دخان | ثنائي |
| تناول الكحول | ميزة ذاتية | الكو | ثنائي |
| النشاط البدني | ميزة ذاتية | نشط | ثنائي |
| وجود أو عدم وجود أمراض القلب والأوعية الدموية | المتغير المستهدف | القلب | ثنائي |
تم جمع جميع قيم مجموعة البيانات في وقت الفحص الطبي.



دعونا نتعرف على بياناتنا من خلال إجراء تحليل أولي للبيانات.
# الجزء الأول. تحليل البيانات الأولية
أولاً، سنقوم بتهيئة البيئة:


In [ ]:
# Import all required modules
# Disable warnings
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# Import plotting modules and set up
import seaborn as sns

sns.set()
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker

%matplotlib inline
%config InlineBackend.figure_format = 'retina'


ستستخدم مكتبة `seaborn` للتحليل المرئي، لذا فلنقم بإعداد ذلك أيضًا:


In [ ]:
# Tune the visual settings for figures in `seaborn`
sns.set_context(
    "notebook", font_scale=1.5, rc={"figure.figsize": (11, 8), "axes.titlesize": 18}
)

from matplotlib import rcParams

rcParams["figure.figsize"] = 11, 8


لتبسيط الأمر، سنعمل فقط مع الجزء التدريبي من مجموعة البيانات:


In [ ]:
df = pd.read_csv("../../data/mlbootcamp5_train.csv", sep=";")
print("Dataset size: ", df.shape)
df.head()


سيكون من المفيد إلقاء نظرة خاطفة على قيم المتغيرات لدينا.
 
لنقم بتحويل البيانات إلى تنسيق *طويل* ونوضح عدد قيم الميزات الفئوية باستخدام [`factorplot()`](https://seaborn.pydata.org/generated/seaborn.factorplot.html).


In [ ]:
df_uniques = pd.melt(
    frame=df,
    value_vars=["gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"],
)
df_uniques = (
    pd.DataFrame(df_uniques.groupby(["variable", "value"])["value"].count())
    .sort_index(level=[0, 1])
    .rename(columns={"value": "count"})
    .reset_index()
)

sns.factorplot(
    x="variable", y="count", hue="value", data=df_uniques, kind="bar", size=12
);


يمكننا أن نرى أن الفئات المستهدفة متوازنة. هذا عظيم!
دعونا نقسم مجموعة البيانات حسب القيم المستهدفة. هل يمكنك بالفعل اكتشاف الميزة الأكثر أهمية بمجرد النظر إلى قطعة الأرض؟


In [ ]:
df_uniques = pd.melt(
    frame=df,
    value_vars=["gender", "cholesterol", "gluc", "smoke", "alco", "active"],
    id_vars=["cardio"],
)
df_uniques = (
    pd.DataFrame(df_uniques.groupby(["variable", "value", "cardio"])["value"].count())
    .sort_index(level=[0, 1])
    .rename(columns={"value": "count"})
    .reset_index()
)

sns.factorplot(
    x="variable",
    y="count",
    hue="value",
    col="cardio",
    data=df_uniques,
    kind="bar",
    size=9,
);

يمكنك أن ترى أن توزيع مستويات الكوليسترول والجلوكوز يختلف بشكل كبير حسب قيمة المتغير المستهدف. هل هذه صدفة؟
الآن، لنحسب بعض الإحصائيات الخاصة بالقيم الفريدة للميزة:


In [ ]:
for c in df.columns:
    n = df[c].nunique()
    print(c)
    if n <= 3:
        print(n, sorted(df[c].value_counts().to_dict().items()))
    else:
        print(n)
    print(10 * "-")


وفي النهاية لدينا:
- 5 ميزات رقمية (باستثناء *المعرف*)؛
- 7 ميزات الفئوية.
- 70000 سجل في المجموع.



##1.1. الملاحظات الأساسية



** السؤال 1.1. (نقطة واحدة). كم عدد الرجال والنساء الموجودين في مجموعة البيانات هذه؟ لم يتم تقديم قيم ميزة `gender` (سواء كان الرقم "1" يشير إلى النساء أو الرجال) - اكتشف ذلك من خلال النظر في تحليل الارتفاع، مع افتراض أن الرجال أطول في المتوسط. **
1. 45530 امرأة و 24470 رجلاً
2. 45530 رجلاً و24470 امرأة
3. 45470 امرأة و 24530 رجلاً
4. 45470 رجلاً و24530 امرأة



** السؤال 1.2. (نقطة واحدة). ما هو الجنس الذي يكثر الحديث عن تعاطي الكحول - الرجال أم النساء؟**
1. النساء
2. الرجال



** السؤال 1.3. (نقطة واحدة). ما الفرق بين نسب المدخنين بين الرجال والنساء (مقربة)؟**
1. 4
2. 16
3. 20
4. 24



** السؤال 1.4. (نقطة واحدة). ما هو الفرق بين متوسط ​​قيم العمر للمدخنين وغير المدخنين (بالشهور، مقربًا)؟ ستحتاج إلى معرفة وحدات الميزة `age` في مجموعة البيانات هذه.**
1. 5
2. 10
3. 15
4. 20



##1.2. خرائط المخاطر
### المهمة:



يتوفر على الموقع الإلكتروني للجمعية الأوروبية لأمراض القلب [مقياس SCORE](https://www.escardio.org/Education/Practice-Tools/CVD-prevention-toolbox/SCORE-Risk-Charts). يتم استخدامه لحساب خطر الوفاة بسبب أمراض القلب والأوعية الدموية في السنوات العشر القادمة. ومن هنا:
<img src='../../img/SCORE_CVD_eng.png' width=70%>
دعونا نلقي نظرة على المستطيل العلوي الأيمن، الذي يظهر مجموعة فرعية من الرجال المدخنين الذين تتراوح أعمارهم بين 60 إلى 65 عاما. (هذا ليس واضحا، ولكن القيم في الشكل تمثل الحد الأعلى).نرى القيمة 9 في الزاوية السفلية اليسرى من المستطيل و47 في الزاوية العلوية اليمنى. وهذا يعني أنه بالنسبة للأشخاص في هذه الفئة العمرية من الجنسين الذين يكون ضغطهم الانقباضي أقل من 120، فإن خطر الإصابة بأمراض القلب والأوعية الدموية يقدر بـ 5 مرات أقل من أولئك الذين يعانون من الضغط في هذه الفترة [160،180).
دعونا نحسب نفس النسبة باستخدام بياناتنا.
توضيحات:
- حساب ميزة ``age_years`` - تقريب العمر إلى أقرب عدد من السنوات. بالنسبة لهذه المهمة، حدد فقط الأشخاص الذين تتراوح أعمارهم بين 60 و64 عامًا.
- تختلف فئات مستوى الكوليسترول بين الشكل ومجموعة البيانات الخاصة بنا. التحويل لميزة ``cholesterol`` هو كما يلي: 4 مليمول/لتر $\rightarrow$ 1، 5-7 مليمول/لتر $\rightarrow$ 2، 8 مليمول/لتر $\rightarrow$ 3.


In [ ]:
# You code here


**السؤال 1.5. (2 نقطة). احسب نسبة الأشخاص المصابين بأمراض القلب والأوعية الدموية في الجزأين الموصوفين أعلاه. ما النسبة بين هذين الكسرين؟**
1. 1
2. 2
3. 3
4. 4



##1.3. تحليل مؤشر كتلة الجسم
### المهمة:



إنشاء ميزة جديدة - مؤشر كتلة الجسم ([مؤشر كتلة الجسم](https://en.wikipedia.org/wiki/Body_mass_index)). للقيام بذلك، قم بتقسيم الوزن بالكيلوجرام على مربع الطول بالأمتار. يقال إن قيم مؤشر كتلة الجسم الطبيعية تتراوح من 18.5 إلى 25. 


In [ ]:
# You code here


**السؤال 1.6. (2 نقطة). اختر العبارات الصحيحة:**
1. متوسط مؤشر كتلة الجسم في العينة يقع ضمن نطاق قيم مؤشر كتلة الجسم الطبيعية.
2. مؤشر كتلة الجسم للنساء أعلى في المتوسط ​​منه للرجال.
3. الأشخاص الأصحاء لديهم، في المتوسط، مؤشر كتلة الجسم أعلى من الأشخاص المصابين بأمراض القلب والأوعية الدموية.
4. بالنسبة للرجال الأصحاء الذين لا يشربون الخمر، يكون مؤشر كتلة الجسم أقرب إلى المعيار مقارنة بالنساء الأصحاء الذين لا يشربون الخمر.



##1.4. بيانات التنظيف



### المهمة:
يمكننا أن نرى أن البيانات ليست مثالية. أنه يحتوي على "الأوساخ" وعدم الدقة. سنرى هذا بشكل أفضل عندما نتصور البيانات.قم بتصفية شرائح المرضى التالية (نعتبرها بيانات خاطئة)
- الضغط الانبساطي أعلى من الضغط الانقباضي 
- الارتفاع أقل من 2.5 بالمائة (استخدم `pd.Series.quantile` لحساب هذه القيمة. إذا لم تكن على دراية بالوظيفة، فيرجى قراءة المستندات.)
- أن يكون الطول أكثر من 97.5 بالمئة
- الوزن أقل من 2.5 بالمئة
- الوزن يزيد بشكل صارم عن 97.5 بالمئة
وهذا ليس كل ما يمكننا القيام به لتنظيف هذه البيانات، ولكن هذا يكفي في الوقت الحالي.


In [ ]:
# You code here


**السؤال 1.7. (2 نقطة). ما هي النسبة المئوية للبيانات الأصلية (المقربة) التي تخلصنا منها؟**
1. 8
2. 9
3. 10
4. 11



# الجزء 2. تحليل البيانات المرئية
##2.1. تصور مصفوفة الارتباط
لفهم الميزات بشكل أفضل، يمكنك إنشاء مصفوفة لمعاملات الارتباط بين الميزات. استخدم مجموعة البيانات الأولية (غير المصفاة).
### المهمة:
ارسم مصفوفة ارتباط باستخدام [`heatmap()`](http://seaborn.pydata.org/generated/seaborn.heatmap.html). يمكنك إنشاء المصفوفة باستخدام أدوات `pandas` القياسية مع المعلمات الافتراضية.


In [ ]:
# You code here


** السؤال 2.1. (نقطة واحدة).** أي زوج من الميزات يتمتع بأقوى ارتباط بيرسون بميزة *الجنس*؟
1. أمراض القلب والكوليسترول
2. الارتفاع، الدخان
3. دخان، ألكو
4. الطول والوزن



##2.2. توزيع الطول للرجال والنساء
من خلال استكشافنا للقيم الفريدة سابقًا، نعلم أن الجنس يتم ترميزه بواسطة القيمتين *1* و*2*. على الرغم من أنك لا تعرف كيفية تعيين هذه القيم للجنس، يمكنك معرفة ذلك بيانيًا من خلال النظر إلى متوسط ​​قيم الطول والوزن لكل قيمة في ميزة *الجنس*.
### مهمة:قم بإنشاء مخطط كمان للطول والجنس باستخدام [`violinplot()`](https://seaborn.pydata.org/generated/seaborn.violinplot.html). استخدم المعلمات:
- `hue` للتقسيم حسب الجنس؛
- `scale` لتقييم عدد السجلات لكل جنس.
لكي يتم عرض المخطط بشكل صحيح، تحتاج إلى تحويل `DataFrame` إلى تنسيق *طويل* باستخدام الدالة `melt()` من `pandas`. إليك [مثال](https://stackoverflow.com/a/41575149/3338479) على ذلك كمرجع لك.


In [ ]:
# You code here


** السؤال 2.2. (نقطة واحدة).** أي زوج من الميزات لديه أقوى ارتباط سبيرمان؟
1. الطول والوزن
2. العمر والوزن
3. الكوليسترول، الجلوك
4. أمراض القلب والكولسترول
5. أب_هي، أب_لو
6. سموك، ألكو



##2.3. ارتباط الرتبة
في معظم الحالات، يكون *معامل بيرسون للارتباط الخطي* أكثر من كافٍ لاكتشاف الأنماط في البيانات. 
ولكن دعنا نذهب أبعد من ذلك قليلاً ونحسب [ارتباط الرتبة](https://en.wikipedia.org/wiki/Rank_correlation). سيساعدنا ذلك في تحديد أزواج الميزات التي يكون فيها الترتيب الأدنى في السلسلة المتغيرة لميزة ما يسبق دائمًا الترتيب الأعلى في الميزة الأخرى (ولدينا العكس في حالة الارتباط السلبي).
### المهمة:
قم بحساب ورسم مصفوفة الارتباط باستخدام [معامل ارتباط رتبة سبيرمان](https://en.wikipedia.org/wiki/Spearman%27s_rank_correlation_coefficient).


In [ ]:
# You code here


**السؤال 2.3. (نقطة واحدة).** لماذا ترتبط هذه الميزات ارتباطًا قويًا بالرتبة؟
1. عدم الدقة في البيانات (أخطاء الحصول على البيانات).
2. العلاقة خاطئة، ولا ينبغي أن تكون هذه الميزات مرتبطة ببعضها البعض.
3. طبيعة البيانات.



##2.4. العمر
في السابق، قمنا بحساب عمر المستجيبين بالسنوات وقت الفحص.



### المهمة:قم بإنشاء *مخطط العد* باستخدام [`countplot()`](http://seaborn.pydata.org/generated/seaborn.countplot.html) مع تحديد العمر على المحور *X* وعدد الأشخاص على المحور *Y*. يجب أن تحتوي قطعة الأرض الناتجة على عمودين لكل عمر، بما يتوافق مع عدد الأشخاص لكل فئة *كارديو* في ذلك العمر.


In [ ]:
# You code here


**السؤال 2.4. (نقطة واحدة).** ما هو أصغر عمر يكون فيه عدد الأشخاص المصابين بأمراض القلب والأوعية الدموية أكبر من عدد الأشخاص غير المصابين بأمراض القلب والأوعية الدموية؟
1. 44
2. 55
3. 64
4. 70